<a href="https://colab.research.google.com/github/jad-r-s/Jad_INFO4670_Fall2026/blob/main/JadSaboune_Week5_INFO4670_Assignment2_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [2]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [4]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = students["study_hours_reported"].isna().sum()

**Your justification (1–2 sentences):** I would use Han's method of ignoring/flagging the tuple because the 255 missing values are likely MNAR, students who skip self-reporting are probably the ones struggling, so filling them with the mean would hide that pattern instead of revealing it, and reporting the honest n = 1,772 is more trustworthy than a biased fill.

In [5]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [6]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
housing_map = {
    "off-campus": "off-campus",
    "off campus": "off-campus",   # missing hyphen variant
    "on-campus": "on-campus",
    "with family": "with family",
}

students["housing_clean"] = (
    students["housing"]
    .str.strip()
    .str.lower()
    .map(housing_map)
)

In [7]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'off-campus': 945, 'on-campus': 590, 'with family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [19]:
# TODO
impossible_mask = (students["age"] < 15) | (students["age"] > 100)
impossible_ages = sorted(students.loc[impossible_mask, "age"].unique())
n_neg_work = (students["work_hours_per_week"] < 0).sum()

In [20]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [25]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates().drop_duplicates(subset="student_id", keep="first")

**Why not de-dupe enroll / activity? (1 sentence):** enroll and activity are tables where one student is meant to appear on many rows (one row per enrollment, one row per student-week), so a repeated student_id there reflects the correct table grain rather than an error, and deduplicating by ID would wrongly erase real enrollments and weekly records.

In [26]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [30]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
students_dedup["id_num"] = students_dedup["student_id"].str.replace("NU-", "", regex=False).astype(int)

activity_summary = (
    activity
    .assign(id_num=activity["student_id"].str.replace("NU-", "", regex=False).astype(int))
    .groupby("id_num")["minutes_active"]
    .sum()
    .reset_index(name="total_minutes_active")
)

analysis = students_dedup.merge(activity_summary, on="id_num", how="left")

In [31]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [34]:
# TODO
matched = enroll["sid"].isin(analysis["id_num"]).sum()

In [35]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [36]:
# TODO
naive_nat_count = pd.to_datetime(students_dedup["enrollment_date"], errors="coerce").isna().sum()

dates_parsed = pd.to_datetime(students_dedup["enrollment_date"], format="mixed", errors="coerce")

parsed_nat_count = dates_parsed.isna().sum()

In [37]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [38]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
analysis["study_z"] = (analysis["study_hours_reported"] - analysis["study_hours_reported"].mean()) / analysis["study_hours_reported"].std()

analysis["gpa_band"] = pd.cut(analysis["final_gpa"], bins=[-0.01, 1, 2, 3, 4], labels=["0-1", "1-2", "2-3", "3-4"])

In [39]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [40]:
# TODO: write analysis to northgate_clean.csv
analysis.to_csv("northgate_clean.csv", index=False)

In [41]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**

- Missing values → left study_hours_reported's 255 blanks unfilled (flagged, not imputed) because the missingness looks MNAR, and imputing would silently hide the pattern of struggling students.
- Housing → standardized 4 raw spellings (off-campus, off campus, on-campus, with family) into 3 real groups, since the extra variant was just a text inconsistency, not a real category.
- Impossible values → treated ages outside a plausible 15–100 range (-22, 0, 1, 3, 199, 220) and negative work_hours_per_week (4 rows) as errors, since a person can't have a negative or near-zero college-student age or negative work hours.
- Duplicates → dropped 18 exact duplicate rows plus collapsed 9 near-duplicate pairs by student_id (2,027 → 2,000 rows), since student_records should hold one row per student.
- Key standardization → stripped the "NU-" prefix and cast to int to build a common numeric key, since student_records/weekly_activity stored IDs as text while course_enrollments stored them as plain numbers.
- Dates → parsed enrollment_date with format="mixed" instead of a naive pd.to_datetime, since the naive parse silently turned 1,470 of 2,027 valid dates into NaT.


### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [42]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
mean_gpa = analysis["final_gpa"].mean()

housing_gpa = analysis.groupby("housing_clean")["final_gpa"].mean()

In [43]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
